In [6]:
# spark session
from pyspark.sql import SparkSession

spark=(
    SparkSession
    .builder
    .appName("Spark Introduction")
    .master("local[*]")
    .getOrCreate()
)

In [7]:
spark

In [8]:
emp_data=[
["001","101","John Doe","30","Male","50000","2015-01-01"],
["002","101","Jane Smith","25","Female","45000","2016-02-15"],
["003","102","Bob Brown","35","Male","50000","2014-05-01"],
["004","102","Alice Lee","28","Female","25000","2017-09-30"],
["005","103","Jack Chan","40","Male","60000","2013-04-01"],
["006","103","Jill Wong","32","Female","52000","2018-07-01"],
["007","104","Tom Tan","33","Male","50000","2016-06-01"],
["008","104","Lisa Lee","27","Female","47000","2014-09-30"],
["009","105","George Wang","34","Male","57000","2018-04-01"],
["010","106","Steven Chan","36","Male","62000","2017-06-01"],
]

emp_Schema="employee_id string, department_id string, name string,age string, gender string, salary string, hire_date string"

In [11]:
# create emp dataframe
emp=spark.createDataFrame(data=emp_data, schema=emp_Schema)

In [12]:
# check no. of partitions
emp.rdd.getNumPartitions()

12

In [13]:
# show data (Actions)
emp.show()

+-----------+-------------+-----------+---+------+------+----------+
|employee_id|department_id|       name|age|gender|salary| hire_date|
+-----------+-------------+-----------+---+------+------+----------+
|        001|          101|   John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101| Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|  Bob Brown| 35|  Male| 50000|2014-05-01|
|        004|          102|  Alice Lee| 28|Female| 25000|2017-09-30|
|        005|          103|  Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|  Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          104|    Tom Tan| 33|  Male| 50000|2016-06-01|
|        008|          104|   Lisa Lee| 27|Female| 47000|2014-09-30|
|        009|          105|George Wang| 34|  Male| 57000|2018-04-01|
|        010|          106|Steven Chan| 36|  Male| 62000|2017-06-01|
+-----------+-------------+-----------+---+------+------+----------+



In [14]:
#write our first transformation (Emp salary > 50000)
emp_final= emp.where("salary> 50000")


In [15]:
# validate number of partitions
emp_final.rdd.getNumPartitions()

12

In [16]:
# write data as CSV output (action)
emp_final.write.format("csv").save("data/output/emp.csv")

In [21]:
#schema for emp
emp.schema

StructType([StructField('employee_id', StringType(), True), StructField('department_id', StringType(), True), StructField('name', StringType(), True), StructField('age', StringType(), True), StructField('gender', StringType(), True), StructField('salary', StringType(), True), StructField('hire_date', StringType(), True)])

In [24]:
#small example for schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema_string="name string, age int"
schema_spark= StructType([
    StructField("name", StringType(),True),
    StructField("age", IntegerType(), True)
])

In [32]:
# columns and expression
from pyspark.sql.functions import col, expr
#emp.salary
emp["salary"]

Column<'salary'>

In [33]:
#select columns employee_id, name, age, salary from emp

emp_filtered= emp.select(col("employee_id"), expr("name"), emp.age, emp.salary)

In [34]:
emp_filtered.show()

+-----------+-----------+---+------+
|employee_id|       name|age|salary|
+-----------+-----------+---+------+
|        001|   John Doe| 30| 50000|
|        002| Jane Smith| 25| 45000|
|        003|  Bob Brown| 35| 50000|
|        004|  Alice Lee| 28| 25000|
|        005|  Jack Chan| 40| 60000|
|        006|  Jill Wong| 32| 52000|
|        007|    Tom Tan| 33| 50000|
|        008|   Lisa Lee| 27| 47000|
|        009|George Wang| 34| 57000|
|        010|Steven Chan| 36| 62000|
+-----------+-----------+---+------+



In [35]:
# using expr for select
# select employee_id as emp_id, name, cast(age as int) as age, salary from emp_filtered
emp_casted= emp_filtered.select(expr("employee_id as emp_id"), emp.name, expr("cast(age as int) as age"), emp.salary)

In [36]:
emp_casted.show()

+------+-----------+---+------+
|emp_id|       name|age|salary|
+------+-----------+---+------+
|   001|   John Doe| 30| 50000|
|   002| Jane Smith| 25| 45000|
|   003|  Bob Brown| 35| 50000|
|   004|  Alice Lee| 28| 25000|
|   005|  Jack Chan| 40| 60000|
|   006|  Jill Wong| 32| 52000|
|   007|    Tom Tan| 33| 50000|
|   008|   Lisa Lee| 27| 47000|
|   009|George Wang| 34| 57000|
|   010|Steven Chan| 36| 62000|
+------+-----------+---+------+



In [37]:
emp_casted.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: string (nullable = true)



In [38]:
emp_casted_1= emp_filtered.selectExpr("employee_id as emp_id","name", "cast(age as int) as age","salary")

In [39]:
emp_casted_1.show()

+------+-----------+---+------+
|emp_id|       name|age|salary|
+------+-----------+---+------+
|   001|   John Doe| 30| 50000|
|   002| Jane Smith| 25| 45000|
|   003|  Bob Brown| 35| 50000|
|   004|  Alice Lee| 28| 25000|
|   005|  Jack Chan| 40| 60000|
|   006|  Jill Wong| 32| 52000|
|   007|    Tom Tan| 33| 50000|
|   008|   Lisa Lee| 27| 47000|
|   009|George Wang| 34| 57000|
|   010|Steven Chan| 36| 62000|
+------+-----------+---+------+



In [40]:
# filter emp based on Age >30
#select emp_id, name, age, salary from emp_casted where age>30

emp_final=emp_casted.select("emp_id", "name","age","salary").where("age>30")

In [41]:
emp_final.show()

+------+-----------+---+------+
|emp_id|       name|age|salary|
+------+-----------+---+------+
|   003|  Bob Brown| 35| 50000|
|   005|  Jack Chan| 40| 60000|
|   006|  Jill Wong| 32| 52000|
|   007|    Tom Tan| 33| 50000|
|   009|George Wang| 34| 57000|
|   010|Steven Chan| 36| 62000|
+------+-----------+---+------+



In [42]:
# write data back as csv (action)
emp_final.write.format("csv").save("data/ouput/2/emp.csv")

In [44]:
#casting columns
# select employee)id, name, age, cast(salary as double) as salary from emp
from pyspark.sql.functions import col, cast

emp_casted= emp.select("employee_id", "name","age",col("salary").cast("double"))

In [45]:
emp_casted.show()

+-----------+-----------+---+-------+
|employee_id|       name|age| salary|
+-----------+-----------+---+-------+
|        001|   John Doe| 30|50000.0|
|        002| Jane Smith| 25|45000.0|
|        003|  Bob Brown| 35|50000.0|
|        004|  Alice Lee| 28|25000.0|
|        005|  Jack Chan| 40|60000.0|
|        006|  Jill Wong| 32|52000.0|
|        007|    Tom Tan| 33|50000.0|
|        008|   Lisa Lee| 27|47000.0|
|        009|George Wang| 34|57000.0|
|        010|Steven Chan| 36|62000.0|
+-----------+-----------+---+-------+



In [46]:
emp_casted.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)



In [47]:
#Adding columns
emp_taxed=emp_casted.withColumn("tax",col("salary")*0.2)


In [48]:
emp_taxed.show()

+-----------+-----------+---+-------+-------+
|employee_id|       name|age| salary|    tax|
+-----------+-----------+---+-------+-------+
|        001|   John Doe| 30|50000.0|10000.0|
|        002| Jane Smith| 25|45000.0| 9000.0|
|        003|  Bob Brown| 35|50000.0|10000.0|
|        004|  Alice Lee| 28|25000.0| 5000.0|
|        005|  Jack Chan| 40|60000.0|12000.0|
|        006|  Jill Wong| 32|52000.0|10400.0|
|        007|    Tom Tan| 33|50000.0|10000.0|
|        008|   Lisa Lee| 27|47000.0| 9400.0|
|        009|George Wang| 34|57000.0|11400.0|
|        010|Steven Chan| 36|62000.0|12400.0|
+-----------+-----------+---+-------+-------+



In [49]:
#Literals
from pyspark.sql.functions import lit

emp_new_cols= emp_taxed.withColumn("columnOne", lit(1)).withColumn("columnTwo", lit("two"))

In [50]:
emp_new_cols.show()

+-----------+-----------+---+-------+-------+---------+---------+
|employee_id|       name|age| salary|    tax|columnOne|columnTwo|
+-----------+-----------+---+-------+-------+---------+---------+
|        001|   John Doe| 30|50000.0|10000.0|        1|      two|
|        002| Jane Smith| 25|45000.0| 9000.0|        1|      two|
|        003|  Bob Brown| 35|50000.0|10000.0|        1|      two|
|        004|  Alice Lee| 28|25000.0| 5000.0|        1|      two|
|        005|  Jack Chan| 40|60000.0|12000.0|        1|      two|
|        006|  Jill Wong| 32|52000.0|10400.0|        1|      two|
|        007|    Tom Tan| 33|50000.0|10000.0|        1|      two|
|        008|   Lisa Lee| 27|47000.0| 9400.0|        1|      two|
|        009|George Wang| 34|57000.0|11400.0|        1|      two|
|        010|Steven Chan| 36|62000.0|12400.0|        1|      two|
+-----------+-----------+---+-------+-------+---------+---------+



In [51]:
#Renaming Columns
emp_1= emp_new_cols.withColumnRenamed("employee_id", "emp_id")

In [52]:
emp_1.show()

+------+-----------+---+-------+-------+---------+---------+
|emp_id|       name|age| salary|    tax|columnOne|columnTwo|
+------+-----------+---+-------+-------+---------+---------+
|   001|   John Doe| 30|50000.0|10000.0|        1|      two|
|   002| Jane Smith| 25|45000.0| 9000.0|        1|      two|
|   003|  Bob Brown| 35|50000.0|10000.0|        1|      two|
|   004|  Alice Lee| 28|25000.0| 5000.0|        1|      two|
|   005|  Jack Chan| 40|60000.0|12000.0|        1|      two|
|   006|  Jill Wong| 32|52000.0|10400.0|        1|      two|
|   007|    Tom Tan| 33|50000.0|10000.0|        1|      two|
|   008|   Lisa Lee| 27|47000.0| 9400.0|        1|      two|
|   009|George Wang| 34|57000.0|11400.0|        1|      two|
|   010|Steven Chan| 36|62000.0|12400.0|        1|      two|
+------+-----------+---+-------+-------+---------+---------+



In [53]:
#column name with spaces
emp_2= emp_new_cols.withColumnRenamed("ColumnTwo", "Column Two")

In [54]:
emp_2.show()

+-----------+-----------+---+-------+-------+---------+----------+
|employee_id|       name|age| salary|    tax|columnOne|Column Two|
+-----------+-----------+---+-------+-------+---------+----------+
|        001|   John Doe| 30|50000.0|10000.0|        1|       two|
|        002| Jane Smith| 25|45000.0| 9000.0|        1|       two|
|        003|  Bob Brown| 35|50000.0|10000.0|        1|       two|
|        004|  Alice Lee| 28|25000.0| 5000.0|        1|       two|
|        005|  Jack Chan| 40|60000.0|12000.0|        1|       two|
|        006|  Jill Wong| 32|52000.0|10400.0|        1|       two|
|        007|    Tom Tan| 33|50000.0|10000.0|        1|       two|
|        008|   Lisa Lee| 27|47000.0| 9400.0|        1|       two|
|        009|George Wang| 34|57000.0|11400.0|        1|       two|
|        010|Steven Chan| 36|62000.0|12400.0|        1|       two|
+-----------+-----------+---+-------+-------+---------+----------+



In [55]:
#Remove column
emp_dropped= emp_new_cols.drop("ColumnTwo")

In [56]:
emp_dropped.show()

+-----------+-----------+---+-------+-------+---------+
|employee_id|       name|age| salary|    tax|columnOne|
+-----------+-----------+---+-------+-------+---------+
|        001|   John Doe| 30|50000.0|10000.0|        1|
|        002| Jane Smith| 25|45000.0| 9000.0|        1|
|        003|  Bob Brown| 35|50000.0|10000.0|        1|
|        004|  Alice Lee| 28|25000.0| 5000.0|        1|
|        005|  Jack Chan| 40|60000.0|12000.0|        1|
|        006|  Jill Wong| 32|52000.0|10400.0|        1|
|        007|    Tom Tan| 33|50000.0|10000.0|        1|
|        008|   Lisa Lee| 27|47000.0| 9400.0|        1|
|        009|George Wang| 34|57000.0|11400.0|        1|
|        010|Steven Chan| 36|62000.0|12400.0|        1|
+-----------+-----------+---+-------+-------+---------+



In [57]:
#Filter data
emp_filtered= emp_dropped.where("tax>10000")

In [58]:
emp_filtered.show()

+-----------+-----------+---+-------+-------+---------+
|employee_id|       name|age| salary|    tax|columnOne|
+-----------+-----------+---+-------+-------+---------+
|        005|  Jack Chan| 40|60000.0|12000.0|        1|
|        006|  Jill Wong| 32|52000.0|10400.0|        1|
|        009|George Wang| 34|57000.0|11400.0|        1|
|        010|Steven Chan| 36|62000.0|12400.0|        1|
+-----------+-----------+---+-------+-------+---------+



In [59]:
#limit data
emp_limit= emp_filtered.limit(5)

In [60]:
emp_limit.show()

+-----------+-----------+---+-------+-------+---------+
|employee_id|       name|age| salary|    tax|columnOne|
+-----------+-----------+---+-------+-------+---------+
|        005|  Jack Chan| 40|60000.0|12000.0|        1|
|        006|  Jill Wong| 32|52000.0|10400.0|        1|
|        009|George Wang| 34|57000.0|11400.0|        1|
|        010|Steven Chan| 36|62000.0|12400.0|        1|
+-----------+-----------+---+-------+-------+---------+

